In [3]:
import numpy as np 
from typing import Tuple, List

In [4]:
import numpy as np
from typing import Tuple

def linear_mnudft_dataset(
    n_samples: int,
    n_positions: int,
    n_frequencies: int,
    seed: int,
    noise_scale: float = 0.0,
    margin: float = 0.3,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:

    rng = np.random.default_rng(seed)

    # ------------------------------------------------------------
    # Fixed 2D spatial geometry p_d ∈ R^2
    # ------------------------------------------------------------
    angles = np.linspace(0.0, 2.0 * np.pi, n_positions, endpoint=False)

    positions = np.stack(
        [np.cos(angles), np.sin(angles)],
        axis=1
    ).astype(np.float64)

    # ------------------------------------------------------------
    # Complex linear separator in spectral space
    # The input signal remains strictly real-valued.
    # Only the MNUDFT coefficients become complex.
    # ------------------------------------------------------------
    w = rng.normal(size=n_frequencies) + 1j * rng.normal(size=n_frequencies)

    # ------------------------------------------------------------
    # Storage
    # ------------------------------------------------------------
    x_list = []
    f_list = []
    score_list = []

    # ------------------------------------------------------------
    # Rejection sampling loop
    # ------------------------------------------------------------
    while len(x_list) < n_samples:

        # --------------------------------------------------------
        # 1. Generate REAL-valued reflectivity signal
        # x(p_d) ∈ R
        # --------------------------------------------------------
        x = rng.normal(size=n_positions)

        # Normalize
        x /= np.max(np.abs(x)) + 1e-12

        # --------------------------------------------------------
        # 2. Generate nonuniform 2D frequency queries
        # f_k ∈ R^2
        # --------------------------------------------------------
        # f_pairs = rng.uniform(
        #     low=-1.0,
        #     high=1.0,
        #     size=(n_frequencies, 2)
        # )

        f_vals = rng.uniform(low=-1.0, high=1.0, size=n_frequencies)
        f_pairs = np.stack([f_vals, f_vals], axis=1)

        # --------------------------------------------------------
        # 3. Evaluate complex MNUDFT
        #
        # X_k = Σ_d x_d exp(-i 2π p_d · f_k)
        #
        # The signal x is real-valued.
        # The Fourier response X is complex-valued.
        # --------------------------------------------------------
        X = np.zeros(n_frequencies, dtype=np.complex128)

        for k in range(n_frequencies):

            phase = 2.0 * np.pi * (positions @ f_pairs[k])

            kernel = np.exp(-1j * phase)

            X[k] = np.sum(x * kernel)

        # --------------------------------------------------------
        # 4. Linear separator in spectral space
        # --------------------------------------------------------
        score = np.real(np.sum(w * X))

        if noise_scale > 0.0:
            score += noise_scale * rng.normal()

        # --------------------------------------------------------
        # 5. Margin filtering
        # --------------------------------------------------------
        if np.abs(score) >= margin:

            # x is already strictly real-valued
            x_list.append(x.astype(np.float64))

            # Flatten frequency pairs:
            # [fx0, fy0, fx1, fy1, ...]
            f_flat = np.zeros(2 * n_frequencies)

            f_flat[0::2] = f_pairs[:, 0]
            f_flat[1::2] = f_pairs[:, 1]

            f_list.append(f_flat.astype(np.float64))

            score_list.append(score)

    # ------------------------------------------------------------
    # Final arrays
    # ------------------------------------------------------------
    x_data = np.array(x_list, dtype=np.float64)

    f_data = np.array(f_list, dtype=np.float64)

    scores = np.array(score_list, dtype=np.float64)

    # ------------------------------------------------------------
    # Balanced binary labels
    # ------------------------------------------------------------
    threshold = np.median(scores)

    y = (scores > threshold).astype(np.int64)

    return x_data, f_data, y, scores


def raw_sar_synthetic_dataset(
    n_samples: int,
    n_qubits: int,
    seed: int,
    noise_scale: float = 0.03,
    dominance_ratio: float = 0.55,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Synthetic raw-SAR task with labels defined after image reconstruction.

    The raw input x stores real/imag parts of complex measurements at K=n_qubits/2
    non-uniform query locations. The paired input f stores the flattened 2D query
    coordinates [fx0, fy0, fx1, fy1, ...]. Labels are extracted from a simple
    inverse reconstruction by counting dominant reflectors in the reconstructed image.
    """
    if n_qubits < 2 or (n_qubits % 2) != 0:
        raise ValueError("n_qubits must be an even integer >= 2")

    rng = np.random.default_rng(seed)
    n_measurements = n_qubits // 2
    n_scene_positions = min(3, n_measurements)

    if n_scene_positions == 1:
        counts_by_label = {0: np.array([0], dtype=np.int64), 1: np.array([1], dtype=np.int64)}
    elif n_scene_positions == 2:
        counts_by_label = {0: np.array([1], dtype=np.int64), 1: np.array([2], dtype=np.int64)}
    else:
        counts_by_label = {0: np.array([2], dtype=np.int64), 1: np.array([3], dtype=np.int64)}

    label_by_count = {
        int(count): int(label)
        for label, counts in counts_by_label.items()
        for count in counts
    }

    if n_scene_positions == 1:
        scene_positions = np.array([[0.0, 0.0]], dtype=np.float64)
    else:
        axis = np.linspace(-0.75, 0.75, n_scene_positions, dtype=np.float64)
        scene_positions = np.stack([axis, 0.35 * np.sin(np.pi * axis)], axis=1).astype(np.float64)

    x_list: List[np.ndarray] = []
    f_list: List[np.ndarray] = []
    y_list: List[int] = []

    if n_measurements == 1:
        base_query_pairs = np.array([[0.35, -0.25]], dtype=np.float64)
    else:
        base_axis = np.linspace(-0.8, 0.8, n_measurements, dtype=np.float64)
        base_query_pairs = np.stack([base_axis, base_axis[::-1]], axis=1).astype(np.float64)

    while len(x_list) < n_samples:
        desired_label = len(x_list) % 2
        active_count = int(rng.choice(counts_by_label[desired_label]))

        reflectivity = np.zeros(n_scene_positions, dtype=np.float64)
        if active_count > 0:
            support = rng.choice(n_scene_positions, size=active_count, replace=False)
            reflectivity[support] = rng.uniform(0.8, 1.2, size=active_count)

        query_pairs = np.clip(
            base_query_pairs + 0.08 * rng.normal(size=(n_measurements, 2)),
            -1.0,
            1.0,
        ).astype(np.float64)
        phases = 2.0 * np.pi * (scene_positions @ query_pairs.T)
        sensing_matrix = np.exp(-1j * phases).T
        if np.linalg.cond(sensing_matrix) > 20.0:
            continue

        measurements = sensing_matrix @ reflectivity

        if noise_scale > 0.0 and active_count > 0:
            noise = rng.normal(size=n_measurements) + 1j * rng.normal(size=n_measurements)
            measurements = measurements + noise_scale * noise / np.sqrt(2.0)

        reconstruction = np.abs(np.linalg.pinv(sensing_matrix) @ measurements)
        peak = float(np.max(reconstruction))
        if peak <= 1e-8:
            dominant_count = 0
        else:
            dominant_count = int(np.sum(reconstruction >= dominance_ratio * peak))
        label = label_by_count.get(dominant_count)
        if label is None:
            continue
        if label != desired_label:
            continue

        x_features = np.concatenate([measurements.real, measurements.imag]).astype(np.float64)
        f_features = np.empty(n_qubits, dtype=np.float64)
        f_features[0::2] = query_pairs[:, 0]
        f_features[1::2] = query_pairs[:, 1]

        x_list.append(x_features)
        f_list.append(f_features)
        y_list.append(label)

    x_data = np.asarray(x_list, dtype=np.float64)
    x_data = (x_data - x_data.mean(axis=0)) / (x_data.std(axis=0) + 1e-8)

    f_data = np.asarray(f_list, dtype=np.float64)
    y = np.asarray(y_list, dtype=np.int64)

    order = rng.permutation(n_samples)
    return x_data[order], f_data[order], y[order]

x,f,y = raw_sar_synthetic_dataset( n_samples=10,
    n_qubits = 6,
    seed = 42,
    noise_scale = 0.1,
    dominance_ratio = 0.55,)

# plt.hist(y)

In [5]:
# x,f,y, scores=linear_mnudft_dataset(
#     n_samples=1000,
#     n_positions=3,
#     n_frequencies=3,
#     seed=42)

In [6]:
# from matplotlib import pyplot as plt
# plt.hist(scores, bins=50)

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

def simple_linear_classifier(X, y):
    """
    Trains a basic Logistic Regression model and returns the 
    trained model along with its accuracy on a test set.
    """
    # 1. Split the data into training and testing sets (80/20 split)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # 2. Initialize the Linear Classifier
    # We use 'lbfgs' solver as it's the default and robust for small/medium data
    model = LogisticRegression(solver='lbfgs', max_iter=1000)

    # 3. Fit the model to the training data
    model.fit(X_train, y_train)

    # 4. Evaluate the model
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    return model, accuracy

trained_model, score = simple_linear_classifier(x, y)

print(f"Model trained! Accuracy: {score * 100:.2f}%")

trained_model, score = simple_linear_classifier(np.concatenate((x, f), axis=1), y)

print(f"Model trained! Accuracy: {score * 100:.2f}%")

Model trained! Accuracy: 100.00%
Model trained! Accuracy: 100.00%


In [9]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


def count_gb_trainable_parameters(model):
    """Return a practical parameter count for a fitted GradientBoostingClassifier.

    We report:
    - internal split params: feature + threshold per internal node
    - leaf value params: one prediction value per leaf
    """
    total_internal_nodes = 0
    total_leaves = 0

    for est in model.estimators_.ravel():
        tree = est.tree_
        total_internal_nodes += tree.node_count - tree.n_leaves
        total_leaves += tree.n_leaves

    split_params = 2 * total_internal_nodes  # (feature, threshold)
    leaf_params = total_leaves
    total_params = split_params + leaf_params

    return {
        "total_params": int(total_params),
        "split_params": int(split_params),
        "leaf_params": int(leaf_params),
        "internal_nodes": int(total_internal_nodes),
        "leaves": int(total_leaves),
    }


def simple_linear_classifier(X, y):
    """
    Trains a basic Logistic Regression model and returns the
    trained model along with its accuracy on a test set.
    """
    # 1. Split the data into training and testing sets (80/20 split)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # 2. Initialize the classifier
    model = GradientBoostingClassifier()

    # 3. Fit the model to the training data
    model.fit(X_train, y_train)

    # 4. Evaluate the model
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    return model, accuracy


trained_model, score = simple_linear_classifier(x, y)
param_report = count_gb_trainable_parameters(trained_model)
print(f"Model trained! Accuracy: {score * 100:.2f}%")
print(
    f"Trainable params: {param_report['total_params']} "
    f"(splits={param_report['split_params']}, leaves={param_report['leaf_params']})"
)

trained_model, score = simple_linear_classifier(np.concatenate((x, f), axis=1), y)
param_report = count_gb_trainable_parameters(trained_model)
print(f"Model trained! Accuracy: {score * 100:.2f}%")
print(
    f"Trainable params: {param_report['total_params']} "
    f"(splits={param_report['split_params']}, leaves={param_report['leaf_params']})"
)

Model trained! Accuracy: 100.00%
Trainable params: 400 (splits=200, leaves=200)
Model trained! Accuracy: 100.00%
Trainable params: 400 (splits=200, leaves=200)
